# Week 3 — Deep Learning Models
**Models**: Bidirectional LSTM, Bidirectional GRU, BERT (bert-base-uncased)  
**Environment**: Google Colab with GPU (T4 recommended)


In [30]:
# Verify GPU availability
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

import sys, os
sys.path.insert(0, 'src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
os.makedirs('results/figures', exist_ok=True)

CUDA available: False


## Part A — BiLSTM and BiGRU

In [31]:
%%writefile /content/lstm_model.py
"""
FAST BiLSTM sentiment classifier (OPTIMIZED VERSION)
"""

import os
import time
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader
from torch.optim import Adam

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

# ─────────────────────────────────────────────
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

DATA_PATH = "reviews_30k.csv"
RESULT_CSV = "results/lstm_results.csv"

os.makedirs("results/figures", exist_ok=True)

# ─────────────────────────────────────────────
# ✨ FAST HYPERPARAMS (FIXED)
# ─────────────────────────────────────────────

HYPERPARAMS = {
    "max_vocab": 5000,      # ↓ faster
    "max_len": 80,          # ↓ faster
    "embed_dim": 32,        # ↓ faster
    "hidden_dim": 64,       # ↓ faster
    "batch_size": 128,
    "lr": 1e-3,
    "epochs": 3,
    "n_layers": 2,          # For BiLSTM and BiGRU
    "dropout": 0.5          # For BiLSTM and BiGRU
}

# ─────────────────────────────────────────────
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"[^a-zA-Z\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

# ─────────────────────────────────────────────
def split_data(df):
    X = df["clean_text"].values
    y = df["label"].values

    X_train, X_temp, y_train, y_temp = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    X_val, X_test, y_val, y_test = train_test_split(
        X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
    )

    return X_train, X_val, X_test, y_train, y_val, y_test

# ─────────────────────────────────────────────
class SimpleTokenizer:
    def __init__(self, max_vocab=5000, max_len=80):
        self.max_vocab = max_vocab
        self.max_len = max_len
        self.word2idx = {"<PAD>": 0, "<UNK>": 1}
        self.idx2word = {0: "<PAD>", 1: "<UNK>"}
        self.word_counts = {}
        self.vocab_size = 2

    def build_vocab(self, texts):
        for text in texts:
            for word in text.split():
                self.word_counts[word] = self.word_counts.get(word, 0) + 1

        sorted_words = sorted(self.word_counts.items(), key=lambda x: x[1], reverse=True)
        for word, _ in sorted_words:
            if self.vocab_size < self.max_vocab:
                self.word2idx[word] = self.vocab_size
                self.idx2word[self.vocab_size] = word
                self.vocab_size += 1

    def encode(self, text):
        encoded = [self.word2idx.get(word, 1) for word in text.split()]
        return encoded[:self.max_len] + [0] * (self.max_len - len(encoded))

    def encode_batch(self, texts):
        return torch.tensor([self.encode(text) for text in texts])

# ─────────────────────────────────────────────
class ReviewDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# ─────────────────────────────────────────────
class BiLSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, n_layers, dropout):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.lstm = nn.LSTM(
            embed_dim, hidden_dim, n_layers,
            bidirectional=True, dropout=dropout, batch_first=True
        )
        self.fc = nn.Linear(hidden_dim * 2, 1)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        embedded = self.dropout(self.embedding(x))
        _, (hidden, _) = self.lstm(embedded)
        hidden = self.dropout(torch.cat((hidden[-2,:,:], hidden[-1,:,:]), dim=1))
        return self.fc(hidden)


class BiGRUClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, n_layers, dropout):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.gru = nn.GRU(
            embed_dim, hidden_dim, n_layers,
            bidirectional=True, dropout=dropout, batch_first=True
        )
        self.fc = nn.Linear(hidden_dim * 2, 1)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        embedded = self.dropout(self.embedding(x))
        _, hidden = self.gru(embedded)
        hidden = self.dropout(torch.cat((hidden[-2,:,:], hidden[-1,:,:]), dim=1))
        return self.fc(hidden)

# ─────────────────────────────────────────────
def train_model(model, train_loader, val_loader, hyperparams, model_name="Model"):
    optimizer = Adam(model.parameters(), lr=hyperparams["lr"])
    criterion = nn.BCEWithLogitsLoss()

    history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
    best_val_loss = float("inf")
    best_model_state = None
    patience_counter = 0
    patience = 3 # Early stopping patience

    print(f"Training {model_name}...")
    for epoch in range(1, hyperparams["epochs"] + 1):
        t0 = time.time()
        train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion)
        val_loss, val_acc, _, _ = evaluate_epoch(model, val_loader, criterion)

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)

        print(
            f"Epoch {epoch}/{hyperparams['epochs']} - "
            f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}, "
            f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f} ({time.time() - t0:.0f}s)"
        )

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_model_state = model.state_dict()
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"Early stopping for {model_name}")
                break

    if best_model_state: # Load best model found during training
        model.load_state_dict(best_model_state)

    return history

def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0
    all_preds = []
    all_labels = []
    for X, y in loader:
        X, y = X.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(X).squeeze(1)
        loss = criterion(outputs, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

        preds = (torch.sigmoid(outputs) > 0.5).long()
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(y.cpu().numpy())

    avg_loss = total_loss / len(loader)
    accuracy = accuracy_score(all_labels, all_preds)
    return avg_loss, accuracy

def evaluate_epoch(model, loader, criterion):
    model.eval()
    total_loss = 0
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for X, y in loader:
            X, y = X.to(DEVICE), y.to(DEVICE)
            outputs = model(X).squeeze(1)
            loss = criterion(outputs, y)
            total_loss += loss.item()

            preds = (torch.sigmoid(outputs) > 0.5).long()
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(y.cpu().numpy())

    avg_loss = total_loss / len(loader)
    accuracy = accuracy_score(all_labels, all_preds)
    return avg_loss, accuracy, all_preds, all_labels

def plot_history(history, model_name, save_path):
    plt.figure(figsize=(12, 5))

    plt.subplot(1, 2, 1)
    plt.plot(history["train_loss"], label="Train Loss")
    plt.plot(history["val_loss"], label="Validation Loss")
    plt.title(f"{model_name} Loss Curve")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(history["train_acc"], label="Train Accuracy")
    plt.plot(history["val_acc"], label="Validation Accuracy")
    plt.title(f"{model_name} Accuracy Curve")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.legend()

    plt.tight_layout()
    plt.savefig(save_path)
    plt.close()

# ─────────────────────────────────────────────
# This `evaluate` function appears to be an older version or incomplete,
# as it's not used in the `main` function (which now uses `evaluate_epoch`).
# It's kept for reference but its direct use might not be intended for training flow.
def evaluate(model, test_loader):
    model.eval()
    preds = []
    labels = []

    with torch.no_grad():
        for x, y in test_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            out = torch.sigmoid(model(x)).cpu().numpy()
            preds.extend((out > 0.5).astype(int))
            labels.extend(y.numpy())

    print(classification_report(labels, preds))

    return (
        accuracy_score(labels, preds),
        precision_score(labels, preds),
        recall_score(labels, preds),
        f1_score(labels, preds)
    )

# ─────────────────────────────────────────────
def main():

    df = pd.read_csv(DATA_PATH)

    df = df[df["Score"] != 3]
    df["label"] = (df["Score"] >= 4).astype(int)

    df = df.sample(10000, random_state=42)

    df["clean_text"] = df["Text"].astype(str).apply(clean_text)

    X_train, X_val, X_test, y_train, y_val, y_test = split_data(df)

    tokenizer = SimpleTokenizer(
        HYPERPARAMS["max_vocab"],
        HYPERPARAMS["max_len"]
    )

    tokenizer.build_vocab(X_train)

    train_ds = ReviewDataset(tokenizer.encode_batch(X_train), y_train)
    val_ds = ReviewDataset(tokenizer.encode_batch(X_val), y_val)
    test_ds = ReviewDataset(tokenizer.encode_batch(X_test), y_test)

    train_loader = DataLoader(train_ds, batch_size=HYPERPARAMS["batch_size"], shuffle=True,
                              num_workers=2, pin_memory=True)

    val_loader = DataLoader(val_ds, batch_size=HYPERPARAMS["batch_size"],
                            num_workers=2, pin_memory=True)

    test_loader = DataLoader(test_ds, batch_size=HYPERPARAMS["batch_size"],
                             num_workers=2, pin_memory=True)

    model = BiLSTMClassifier(
        tokenizer.vocab_size,
        HYPERPARAMS["embed_dim"],
        HYPERPARAMS["hidden_dim"],
        HYPERPARAMS["n_layers"], # Added missing argument
        HYPERPARAMS["dropout"]    # Added missing argument
    ).to(DEVICE)

    print("Params:", sum(p.numel() for p in model.parameters()))

    history = train_model(model, train_loader, val_loader, HYPERPARAMS)

    acc, prec, rec, f1 = evaluate(model, test_loader)

    pd.DataFrame([{
        "model": "BiLSTM_FAST",
        "accuracy": acc,
        "precision": prec,
        "recall": rec,
        "f1": f1
    }]).to_csv(RESULT_CSV, index=False)

    print("Saved!")

if __name__ == "__main__":
    main()


Overwriting /content/lstm_model.py


In [32]:
from preprocessing import clean_text, split_data, SimpleTokenizer
import lstm_model # Import the module first
import importlib
importlib.reload(lstm_model) # Then reload it to apply changes
from lstm_model import (
    BiLSTMClassifier, BiGRUClassifier,
    ReviewDataset, train_model, evaluate_epoch, plot_history,
    DEVICE, HYPERPARAMS
)
from torch.utils.data import DataLoader
import torch.nn as nn
from evaluate import compute_metrics, print_report, plot_confusion_matrix

df = pd.read_csv('reviews_30k.csv')

# Add these lines to create the 'label' column
df = df[df["Score"] != 3]
df["label"] = (df["Score"] >= 4).astype(int)

print('Cleaning text ...')
df['clean_text'] = df['Text'].apply(clean_text)

hp = HYPERPARAMS
X_train, X_val, X_test, y_train, y_val, y_test = split_data(df)

tokenizer = SimpleTokenizer(max_vocab=hp['max_vocab'], max_len=hp['max_len'])
tokenizer.build_vocab(X_train)

X_train_enc = tokenizer.encode_batch(X_train)
X_val_enc   = tokenizer.encode_batch(X_val)
X_test_enc  = tokenizer.encode_batch(X_test)

train_loader = DataLoader(ReviewDataset(X_train_enc, y_train), batch_size=hp['batch_size'], shuffle=True)
val_loader   = DataLoader(ReviewDataset(X_val_enc,   y_val),   batch_size=hp['batch_size'])
test_loader  = DataLoader(ReviewDataset(X_test_enc,  y_test),  batch_size=hp['batch_size'])

Using device: cpu
Cleaning text ...
Split sizes  →  train: 19,409  val: 4,160  test: 4,160
Vocabulary built: 5,000 tokens


In [33]:
"""
FAST BiLSTM sentiment classifier (OPTIMIZED VERSION)
"""

import os
import time
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader
from torch.optim import Adam

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

# ─────────────────────────────────────────────
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

DATA_PATH = "reviews_30k.csv"
RESULT_CSV = "results/lstm_results.csv"

os.makedirs("results/figures", exist_ok=True)

# ─────────────────────────────────────────────
# ✨ FAST HYPERPARAMS (FIXED)
# ─────────────────────────────────────────────

HYPERPARAMS = {
    "max_vocab": 5000,      # ↓ faster
    "max_len": 80,          # ↓ faster
    "embed_dim": 32,        # ↓ faster
    "hidden_dim": 64,       # ↓ faster
    "batch_size": 128,
    "lr": 1e-3,
    "epochs": 3,
    "n_layers": 2,          # For BiLSTM and BiGRU
    "dropout": 0.5          # For BiLSTM and BiGRU
}

# ─────────────────────────────────────────────
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"[^a-zA-Z\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

# ─────────────────────────────────────────────
def split_data(df):
    X = df["clean_text"].values
    y = df["label"].values

    X_train, X_temp, y_train, y_temp = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    X_val, X_test, y_val, y_test = train_test_split(
        X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
    )

    return X_train, X_val, X_test, y_train, y_val, y_test

# ─────────────────────────────────────────────
class SimpleTokenizer:
    def __init__(self, max_vocab=5000, max_len=80):
        self.max_vocab = max_vocab
        self.max_len = max_len
        self.word2idx = {"<PAD>": 0, "<UNK>": 1}
        self.idx2word = {0: "<PAD>", 1: "<UNK>"}
        self.word_counts = {}
        self.vocab_size = 2

    def build_vocab(self, texts):
        for text in texts:
            for word in text.split():
                self.word_counts[word] = self.word_counts.get(word, 0) + 1

        sorted_words = sorted(self.word_counts.items(), key=lambda x: x[1], reverse=True)
        for word, _ in sorted_words:
            if self.vocab_size < self.max_vocab:
                self.word2idx[word] = self.vocab_size
                self.idx2word[self.vocab_size] = word
                self.vocab_size += 1

    def encode(self, text):
        encoded = [self.word2idx.get(word, 1) for word in text.split()]
        return encoded[:self.max_len] + [0] * (self.max_len - len(encoded))

    def encode_batch(self, texts):
        return torch.tensor([self.encode(text) for text in texts])

# ─────────────────────────────────────────────
class ReviewDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# ─────────────────────────────────────────────
class BiLSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, n_layers, dropout):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.lstm = nn.LSTM(
            embed_dim, hidden_dim, n_layers,
            bidirectional=True, dropout=dropout, batch_first=True
        )
        self.fc = nn.Linear(hidden_dim * 2, 1)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        embedded = self.dropout(self.embedding(x))
        _, (hidden, _) = self.lstm(embedded)
        hidden = self.dropout(torch.cat((hidden[-2,:,:], hidden[-1,:,:]), dim=1))
        return self.fc(hidden)


class BiGRUClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, n_layers, dropout):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.gru = nn.GRU(
            embed_dim, hidden_dim, n_layers,
            bidirectional=True, dropout=dropout, batch_first=True
        )
        self.fc = nn.Linear(hidden_dim * 2, 1)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        embedded = self.dropout(self.embedding(x))
        _, hidden = self.gru(embedded)
        hidden = self.dropout(torch.cat((hidden[-2,:,:], hidden[-1,:,:]), dim=1))
        return self.fc(hidden)

# ─────────────────────────────────────────────
def train_model(model, train_loader, val_loader, hyperparams, model_name="Model"):
    optimizer = Adam(model.parameters(), lr=hyperparams["lr"])
    criterion = nn.BCEWithLogitsLoss()

    history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
    best_val_loss = float("inf")
    best_model_state = None
    patience_counter = 0
    patience = 3 # Early stopping patience

    print(f"Training {model_name}...")
    for epoch in range(1, hyperparams["epochs"] + 1):
        t0 = time.time()
        train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion)
        val_loss, val_acc, _, _ = evaluate_epoch(model, val_loader, criterion)

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)

        print(
            f"Epoch {epoch}/{hyperparams['epochs']} - "
            f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}, "
            f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f} ({time.time() - t0:.0f}s)"
        )

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_model_state = model.state_dict()
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"Early stopping for {model_name}")
                break

    if best_model_state: # Load best model found during training
        model.load_state_dict(best_model_state)

    return history

def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0
    all_preds = []
    all_labels = []
    for X, y in loader:
        X, y = X.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(X).squeeze(1)
        loss = criterion(outputs, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

        preds = (torch.sigmoid(outputs) > 0.5).long()
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(y.cpu().numpy())

    avg_loss = total_loss / len(loader)
    accuracy = accuracy_score(all_labels, all_preds)
    return avg_loss, accuracy

def evaluate_epoch(model, loader, criterion):
    model.eval()
    total_loss = 0
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for X, y in loader:
            X, y = X.to(DEVICE), y.to(DEVICE)
            outputs = model(X).squeeze(1)
            loss = criterion(outputs, y)
            total_loss += loss.item()

            preds = (torch.sigmoid(outputs) > 0.5).long()
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(y.cpu().numpy())

    avg_loss = total_loss / len(loader)
    accuracy = accuracy_score(all_labels, all_preds)
    return avg_loss, accuracy, all_preds, all_labels

def plot_history(history, model_name, save_path):
    plt.figure(figsize=(12, 5))

    plt.subplot(1, 2, 1)
    plt.plot(history["train_loss"], label="Train Loss")
    plt.plot(history["val_loss"], label="Validation Loss")
    plt.title(f"{model_name} Loss Curve")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(history["train_acc"], label="Train Accuracy")
    plt.plot(history["val_acc"], label="Validation Accuracy")
    plt.title(f"{model_name} Accuracy Curve")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.legend()

    plt.tight_layout()
    plt.savefig(save_path)
    plt.close()

# ─────────────────────────────────────────────
# This `evaluate` function appears to be an older version or incomplete,
# as it's not used in the `main` function (which now uses `evaluate_epoch`).
# It's kept for reference but its direct use might not be intended for training flow.
def evaluate(model, test_loader):
    model.eval()
    preds = []
    labels = []

    with torch.no_grad():
        for x, y in test_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            out = torch.sigmoid(model(x)).cpu().numpy()
            preds.extend((out > 0.5).astype(int))
            labels.extend(y.numpy())

    print(classification_report(labels, preds))

    return (
        accuracy_score(labels, preds),
        precision_score(labels, preds),
        recall_score(labels, preds),
        f1_score(labels, preds)
    )

# ─────────────────────────────────────────────
def main():

    df = pd.read_csv(DATA_PATH)

    df = df[df["Score"] != 3]
    df["label"] = (df["Score"] >= 4).astype(int)

    df = df.sample(10000, random_state=42)

    df["clean_text"] = df["Text"].astype(str).apply(clean_text)

    X_train, X_val, X_test, y_train, y_val, y_test = split_data(df)

    tokenizer = SimpleTokenizer(
        HYPERPARAMS["max_vocab"],
        HYPERPARAMS["max_len"]
    )

    tokenizer.build_vocab(X_train)

    train_ds = ReviewDataset(tokenizer.encode_batch(X_train), y_train)
    val_ds = ReviewDataset(tokenizer.encode_batch(X_val), y_val)
    test_ds = ReviewDataset(tokenizer.encode_batch(X_test), y_test)

    train_loader = DataLoader(train_ds, batch_size=HYPERPARAMS["batch_size"], shuffle=True,
                              num_workers=2, pin_memory=True)

    val_loader = DataLoader(val_ds, batch_size=HYPERPARAMS["batch_size"],
                            num_workers=2, pin_memory=True)

    test_loader = DataLoader(test_ds, batch_size=HYPERPARAMS["batch_size"],
                             num_workers=2, pin_memory=True)

    model = BiLSTMClassifier(
        tokenizer.vocab_size,
        HYPERPARAMS["embed_dim"],
        HYPERPARAMS["hidden_dim"],
        HYPERPARAMS["n_layers"], # Added missing argument
        HYPERPARAMS["dropout"]    # Added missing argument
    ).to(DEVICE)

    print("Params:", sum(p.numel() for p in model.parameters()))

    history = train_model(model, train_loader, val_loader, HYPERPARAMS)

    acc, prec, rec, f1 = evaluate(model, test_loader)

    pd.DataFrame([{
        "model": "BiLSTM_FAST",
        "accuracy": acc,
        "precision": prec,
        "recall": rec,
        "f1": f1
    }]).to_csv(RESULT_CSV, index=False)

    print("Saved!")

if __name__ == "__main__":
    main()


Using device: cpu
Params: 309633
Training Model...


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch 1/3 - Train Loss: 0.4653, Train Acc: 0.8424, Val Loss: 0.4292, Val Acc: 0.8440 (25s)


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch 2/3 - Train Loss: 0.4323, Train Acc: 0.8440, Val Loss: 0.4230, Val Acc: 0.8440 (25s)


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch 3/3 - Train Loss: 0.4282, Train Acc: 0.8440, Val Loss: 0.4089, Val Acc: 0.8440 (26s)


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


              precision    recall  f1-score   support

         0.0       0.00      0.00      0.00       156
         1.0       0.84      1.00      0.92       844

    accuracy                           0.84      1000
   macro avg       0.42      0.50      0.46      1000
weighted avg       0.71      0.84      0.77      1000

Saved!


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [34]:
# ── BiLSTM ──────────────────────────────────────────────────────────────────
lstm_model = BiLSTMClassifier(
    vocab_size=tokenizer.vocab_size,
    embed_dim=hp['embed_dim'],
    hidden_dim=hp['hidden_dim'],
    n_layers=hp['n_layers'],
    dropout=hp['dropout'],
).to(DEVICE)

print(f'BiLSTM parameters: {sum(p.numel() for p in lstm_model.parameters()):,}')
lstm_history = train_model(lstm_model, train_loader, val_loader, hp, 'BiLSTM')
plot_history(lstm_history, 'BiLSTM', 'results/figures/bilstm_loss_curve.png')

BiLSTM parameters: 309,633
Training BiLSTM...
Epoch 1/3 - Train Loss: 0.4437, Train Acc: 0.8358, Val Loss: 0.4148, Val Acc: 0.8447 (57s)
Epoch 2/3 - Train Loss: 0.4061, Train Acc: 0.8447, Val Loss: 0.3876, Val Acc: 0.8447 (56s)
Epoch 3/3 - Train Loss: 0.3736, Train Acc: 0.8455, Val Loss: 0.3949, Val Acc: 0.8481 (57s)


In [35]:
# ── BiGRU ────────────────────────────────────────────────────────────────────
gru_model = BiGRUClassifier(
    vocab_size=tokenizer.vocab_size,
    embed_dim=hp['embed_dim'],
    hidden_dim=hp['hidden_dim'],
    n_layers=hp['n_layers'],
    dropout=hp['dropout'],
).to(DEVICE)

print(f'BiGRU parameters: {sum(p.numel() for p in gru_model.parameters()):,}')
gru_history = train_model(gru_model, train_loader, val_loader, hp, 'BiGRU')
plot_history(gru_history, 'BiGRU', 'results/figures/bigru_loss_curve.png')

BiGRU parameters: 272,257
Training BiGRU...
Epoch 1/3 - Train Loss: 0.4432, Train Acc: 0.8422, Val Loss: 0.4236, Val Acc: 0.8447 (55s)
Epoch 2/3 - Train Loss: 0.4171, Train Acc: 0.8451, Val Loss: 0.4003, Val Acc: 0.8459 (54s)
Epoch 3/3 - Train Loss: 0.3781, Train Acc: 0.8475, Val Loss: 0.3731, Val Acc: 0.8558 (54s)


In [36]:
# Test evaluation for RNN models
criterion = nn.BCEWithLogitsLoss()
rnn_results = []

for name, model in [('BiLSTM', lstm_model), ('BiGRU', gru_model)]:
    _, _, y_pred, y_true = evaluate_epoch(model, test_loader, criterion)
    metrics = compute_metrics(y_true, y_pred)
    print_report(y_true, y_pred, name)
    plot_confusion_matrix(y_true, y_pred, name, f'results/figures/{name.lower()}_confusion.png')
    rnn_results.append({'model': name, **metrics})

pd.DataFrame(rnn_results).to_csv('results/lstm_results.csv', index=False)
print(pd.DataFrame(rnn_results))


Model: BiLSTM
              precision    recall  f1-score   support

    Negative       0.90      0.04      0.08       646
    Positive       0.85      1.00      0.92      3514

    accuracy                           0.85      4160
   macro avg       0.87      0.52      0.50      4160
weighted avg       0.86      0.85      0.79      4160

Confusion matrix saved: results/figures/bilstm_confusion.png

Model: BiGRU
              precision    recall  f1-score   support

    Negative       0.62      0.24      0.35       646
    Positive       0.88      0.97      0.92      3514

    accuracy                           0.86      4160
   macro avg       0.75      0.61      0.64      4160
weighted avg       0.84      0.86      0.83      4160

Confusion matrix saved: results/figures/bigru_confusion.png
    model  accuracy  precision  recall      f1
0  BiLSTM    0.8502     0.8499  0.9991  0.9185
1   BiGRU    0.8594     0.8750  0.9724  0.9211


## Part B — BERT Fine-tuning

In [37]:
# Install if needed (Colab)
# !pip install -q transformers accelerate

from bert_model import (
    BertReviewDataset, train_epoch as bert_train,
    evaluate_epoch as bert_eval, plot_history as bert_plot_history,
    HYPERPARAMS as BERT_HP
)
from transformers import BertTokenizer, BertForSequenceClassification, get_linear_schedule_with_warmup
from torch.optim import AdamW

# Use raw text for BERT
# Create a temporary DataFrame for BERT where 'clean_text' refers to the original 'Text' column
bert_df = df.copy()
bert_df['clean_text'] = bert_df['Text']
X_train_raw, X_val_raw, X_test_raw, y_train_b, y_val_b, y_test_b = split_data(
    bert_df # Pass the modified DataFrame
)

bert_hp = BERT_HP
print('Loading BERT tokenizer ...')
bert_tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

bert_train_ds = BertReviewDataset(X_train_raw, y_train_b, bert_tokenizer, bert_hp['max_len'])
bert_val_ds   = BertReviewDataset(X_val_raw,   y_val_b,   bert_tokenizer, bert_hp['max_len'])
bert_test_ds  = BertReviewDataset(X_test_raw,  y_test_b,  bert_tokenizer, bert_hp['max_len'])

bert_train_loader = DataLoader(bert_train_ds, batch_size=bert_hp['batch_size'], shuffle=True)
bert_val_loader   = DataLoader(bert_val_ds,   batch_size=bert_hp['batch_size'])
bert_test_loader  = DataLoader(bert_test_ds,  batch_size=bert_hp['batch_size'])

print('Loading BERT model ...')
bert_model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2).to(DEVICE)
print(f'BERT parameters: {sum(p.numel() for p in bert_model.parameters()):,}')

Loading BERT tokenizer ...
Loading BERT model ...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


BERT parameters: 109,483,778


In [39]:
import time
import re
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# ─────────────────────────────
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

# ─────────────────────────────
# FAST CONFIG
# ─────────────────────────────
MAX_VOCAB = 5000
MAX_LEN = 80
EMBED_DIM = 32
HIDDEN_DIM = 64
BATCH_SIZE = 128
EPOCHS = 3
LR = 1e-3

# ─────────────────────────────
# DATA
# ─────────────────────────────
df = pd.read_csv("reviews_30k.csv")

df = df[df["Score"] != 3]
df["label"] = (df["Score"] >= 4).astype(int)

df = df.sample(10000, random_state=42)

def clean(text):
    text = str(text).lower()
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"[^a-zA-Z\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["text"] = df["Text"].apply(clean)

X_train, X_temp, y_train, y_temp = train_test_split(
    df["text"].values,
    df["label"].values,
    test_size=0.2,
    stratify=df["label"],
    random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.5,
    stratify=y_temp,
    random_state=42
)

# ─────────────────────────────
# TOKENIZER (simple)
# ─────────────────────────────
word2idx = {"<PAD>": 0, "<UNK>": 1}

freq = {}
for text in X_train:
    for w in text.split():
        freq[w] = freq.get(w, 0) + 1

sorted_words = sorted(freq.items(), key=lambda x: x[1], reverse=True)

for i, (w, _) in enumerate(sorted_words[:MAX_VOCAB-2], 2):
    word2idx[w] = i

def encode(text):
    tokens = [word2idx.get(w, 1) for w in text.split()][:MAX_LEN]
    tokens += [0] * (MAX_LEN - len(tokens))
    return tokens

# ─────────────────────────────
# DATASET
# ─────────────────────────────
class TextDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor([encode(t) for t in X], dtype=torch.long)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, i):
        return self.X[i], self.y[i]

train_loader = DataLoader(TextDataset(X_train, y_train), batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(TextDataset(X_val, y_val), batch_size=BATCH_SIZE, num_workers=2, pin_memory=True)

# ─────────────────────────────
# BiGRU MODEL (FAST)
# ─────────────────────────────
class BiGRU(nn.Module):
    def __init__(self):
        super().__init__()

        self.embedding = nn.Embedding(len(word2idx), EMBED_DIM, padding_idx=0)

        self.gru = nn.GRU(
            EMBED_DIM,
            HIDDEN_DIM,
            batch_first=True,
            bidirectional=True
        )

        self.fc = nn.Linear(HIDDEN_DIM * 2, 1)
        self.dropout = nn.Dropout(0.3)

    def forward(self, x):
        x = self.embedding(x)
        _, h = self.gru(x)

        h = torch.cat((h[-2], h[-1]), dim=1)
        h = self.dropout(h)

        return self.fc(h).squeeze(1)

model = BiGRU().to(DEVICE)

# ─────────────────────────────
# LOSS / OPTIM
# ─────────────────────────────
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

scaler = torch.cuda.amp.GradScaler()

# ─────────────────────────────
# TRAIN
# ─────────────────────────────
def train_epoch():
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for x, y in train_loader:
        x, y = x.to(DEVICE), y.to(DEVICE)

        optimizer.zero_grad()

        with torch.cuda.amp.autocast():
            out = model(x)
            loss = criterion(out, y)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()

        preds = (torch.sigmoid(out) > 0.5).long()
        correct += (preds == y.long()).sum().item()
        total += y.size(0)

    return total_loss / len(train_loader), correct / total

# ─────────────────────────────
# EVAL
# ─────────────────────────────
@torch.no_grad()
def evaluate():
    model.eval()

    preds_all, labels_all = [], []

    for x, y in val_loader:
        x = x.to(DEVICE)

        out = model(x)
        preds = (torch.sigmoid(out) > 0.5).cpu().numpy()

        preds_all.extend(preds)
        labels_all.extend(y.numpy())

    print("Acc:", accuracy_score(labels_all, preds_all))
    print("Prec:", precision_score(labels_all, preds_all))
    print("Rec:", recall_score(labels_all, preds_all))
    print("F1:", f1_score(labels_all, preds_all))

# ─────────────────────────────
# RUN
# ─────────────────────────────
for epoch in range(EPOCHS):
    t0 = time.time()

    loss, acc = train_epoch()
    evaluate()

    print(f"Epoch {epoch+1} | loss={loss:.4f} acc={acc:.4f} time={time.time()-t0:.1f}s")

Device: cpu


/tmp/ipykernel_33351/1544677340.py:135: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()
/usr/local/lib/python3.12/dist-packages/torch/cuda/amp/grad_scaler.py:31: UserWarning: torch.cuda.amp.GradScaler is enabled, but CUDA is not available.  Disabling.
  super().__init__(
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
/tmp/ipykernel_33351/1544677340.py:151: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
/usr/local/lib/python3.12/dist-packages/torch/cuda/amp/autocast_mode.py:54: UserWarning: CUDA is not available or torch_xla is imported. Disabling autocast.
  super().__init__(


Acc: 0.844
Prec: 0.844
Rec: 1.0
F1: 0.9154013015184381
Epoch 1 | loss=0.4812 acc=0.8310 time=11.0s


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
/tmp/ipykernel_33351/1544677340.py:151: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
/usr/local/lib/python3.12/dist-packages/torch/cuda/amp/autocast_mode.py:54: UserWarning: CUDA is not available or torch_xla is imported. Disabling autocast.
  super().__init__(


Acc: 0.842
Prec: 0.8443775100401606
Rec: 0.9964454976303317
F1: 0.9141304347826087
Epoch 2 | loss=0.4119 acc=0.8440 time=10.6s


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
/tmp/ipykernel_33351/1544677340.py:151: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
/usr/local/lib/python3.12/dist-packages/torch/cuda/amp/autocast_mode.py:54: UserWarning: CUDA is not available or torch_xla is imported. Disabling autocast.
  super().__init__(


Acc: 0.844
Prec: 0.8460764587525151
Rec: 0.9964454976303317
F1: 0.9151251360174102
Epoch 3 | loss=0.3875 acc=0.8450 time=10.2s


In [40]:
import torch
import pandas as pd

bert_model.to(DEVICE)
bert_model.eval()

bert_y_pred = []
bert_y_true = []

loader = bert_test_loader

with torch.inference_mode():
    for batch in loader:
        input_ids = batch["input_ids"].to(DEVICE, non_blocking=True)
        attention_mask = batch["attention_mask"].to(DEVICE, non_blocking=True)
        labels = batch["labels"]

        # AMP inference
        with torch.cuda.amp.autocast():
            logits = bert_model(
                input_ids=input_ids,
                attention_mask=attention_mask
            ).logits

        preds = logits.argmax(dim=1)

        bert_y_pred.append(preds.cpu())
        bert_y_true.append(labels)

bert_y_pred = torch.cat(bert_y_pred).numpy()
bert_y_true = torch.cat(bert_y_true).numpy()

bert_metrics = compute_metrics(bert_y_true, bert_y_pred)

print_report(bert_y_true, bert_y_pred, "BERT")

plot_confusion_matrix(
    bert_y_true,
    bert_y_pred,
    "BERT",
    "results/figures/bert_confusion.png"
)

print(bert_metrics)

pd.DataFrame([{"model": "BERT", **bert_metrics}]).to_csv(
    "results/bert_results.csv",
    index=False
)


Model: BERT
              precision    recall  f1-score   support

    Negative       0.33      0.00      0.00       431
    Positive       0.84      1.00      0.92      2342

    accuracy                           0.84      2773
   macro avg       0.59      0.50      0.46      2773
weighted avg       0.77      0.84      0.77      2773

Confusion matrix saved: results/figures/bert_confusion.png
{'accuracy': 0.8442, 'precision': 0.8448, 'recall': 0.9991, 'f1': 0.9155}


## Error Analysis — BERT

In [41]:
from evaluate import error_analysis
errors = error_analysis(X_test_raw, bert_y_true, bert_y_pred, n_samples=10)
errors.to_csv('results/bert_error_analysis.csv', index=False)

for _, row in errors.head(5).iterrows():
    print(f'[True: {row.true_label}] [Pred: {row.predicted_label}] [{row.error_type}]')
    print(row.text[:200])
    print()

[True: Negative] [Pred: Positive] [False Positive]
First off, the initial arrival:  It came in a bigger box (since other things were ordered as well) when I opened it the 2 yellow bags set neatly and ready for me to eat!  Packaging, no other boxes or 

[True: Negative] [Pred: Positive] [False Positive]
I guess its okay using these pods for latte's but if your looking for a good cup of espresso, i wouldnt recommend.  Its way too light for my taste.

[True: Negative] [Pred: Positive] [False Positive]
I've been trying to give up both Aspartame and High-Fructose Corn Syrup, so I was excited to find Zevia at my local grocery store. I've had stevia one other time, in coffee, and it seemed a little ove

[True: Negative] [Pred: Positive] [False Positive]
Jet Fuel is without a doubt, the most bitter coffee we have tried so far. Odds are very good a severe case of heart burn will closely follow your morning coffee with this stuff. Buyer beware!

[True: Negative] [Pred: Positive] [False Positive]